# VVMV Colab Full Multi-field Rule Mining

Chạy rule mining nặng trên Google Colab/Drive cho logic nhiều trường (`A+B` và `A+B+C -> target`).


## 1. Mount Google Drive hoặc chạy local
Cell này tự bỏ qua mount nếu không chạy trong Google Colab.


In [ ]:
# 1) Mount Google Drive nếu đang chạy trong Google Colab
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
    print('Không phát hiện Google Colab; bỏ qua drive.mount().')
    print('Nếu chạy local/Jupyter, hãy chỉnh PROJECT_DIR ở cell cấu hình bên dưới.')


## 2. Cấu hình và hàm mining
Sửa `PROJECT_DIR`, `DECL_PATH`, `CARGO_PATH` nếu dữ liệu nằm chỗ khác.


In [ ]:

from __future__ import annotations

import itertools
import json
import math
import os
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

# ===== CONFIG =====
# Option A: put files in Google Drive and update PROJECT_DIR.
PROJECT_DIR = Path('/content/drive/MyDrive/ai-vvmv') if globals().get('IN_COLAB', False) else Path('/Volumes/FastVault-Apps/Projects/ai-vvmv')
DECL_PATH = PROJECT_DIR / 'llm-dataset/ai-assets/analytics/declarations_compact.jsonl'
CARGO_PATH = PROJECT_DIR / 'llm-dataset/ai-assets/analytics/cargo_facts.jsonl'
OUT_DIR = PROJECT_DIR / 'llm-dataset/ai-assets/reports'

# Option B: upload files to /content/data and uncomment these lines.
# DECL_PATH = Path('/content/data/declarations_compact.jsonl')
# CARGO_PATH = Path('/content/data/cargo_facts.jsonl')
# OUT_DIR = Path('/content/outputs')

MIN_SUPPORT = 1000
CONFIDENCE = 0.98
MAX_ARITY = 3
MIN_LIFT = 0.02

DECL_RULE_FIELDS = [
    'direction', 'operation_code', 'customs_code', 'export_country',
    'import_country', 'delivery_term', 'currency', 'port_code',
]
DECL_TARGETS = [
    'direction', 'customs_code', 'export_country', 'import_country',
    'delivery_term', 'currency', 'port_code',
]
CARGO_RULE_FIELDS = [
    'direction', 'operation_code', 'customs_code', 'hs2', 'hs4', 'hs6',
    'origin', 'unit', 'tax_code',
]
CARGO_TARGETS = ['direction', 'customs_code', 'origin', 'unit', 'tax_code', 'hs2', 'hs4']


def iter_jsonl(path: Path) -> Iterable[Dict[str, Any]]:
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


def norm(value: Any) -> str:
    if value is None:
        return ''
    return str(value).strip()


def enrich_cargo(rec: Dict[str, Any]) -> Dict[str, str]:
    out = {field: norm(rec.get(field)) for field in CARGO_RULE_FIELDS}
    hs = norm(rec.get('hs_code'))
    out['hs2'] = hs[:2] if len(hs) >= 2 else ''
    out['hs4'] = hs[:4] if len(hs) >= 4 else ''
    out['hs6'] = hs[:6] if len(hs) >= 6 else ''
    return out


def condition_key(values: Dict[str, str], fields: Tuple[str, ...]) -> Tuple[Tuple[str, str], ...] | None:
    pairs = []
    for field in fields:
        value = values.get(field, '')
        if not value:
            return None
        pairs.append((field, value))
    return tuple(pairs)


def specificity_score(rule: Dict[str, Any]) -> Tuple[int, float, int]:
    return (len(rule['conditions']), rule['confidence'], rule['support'])


def mine_rules(path: Path, condition_fields: List[str], targets: List[str], dataset: str,
               min_support: int, threshold: float, max_arity: int) -> List[Dict[str, Any]]:
    condition_combos = []
    for arity in range(2, max_arity + 1):
        condition_combos.extend(itertools.combinations(condition_fields, arity))
    print(f'{dataset}: condition combos={len(condition_combos)}')

    started = time.time()
    condition_counts: Counter = Counter()
    for row_no, rec in enumerate(iter_jsonl(path), 1):
        values = enrich_cargo(rec) if dataset == 'cargo' else {field: norm(rec.get(field)) for field in condition_fields}
        for combo in condition_combos:
            cond = condition_key(values, combo)
            if cond:
                condition_counts[cond] += 1
        if row_no % 250_000 == 0:
            print(f'{dataset}: pass1 rows={row_no:,} eligible_so_far={sum(1 for v in condition_counts.values() if v >= min_support):,} elapsed={time.time()-started:.1f}s')

    eligible_conditions = {cond for cond, count in condition_counts.items() if count >= min_support}
    print(f'{dataset}: eligible_conditions={len(eligible_conditions):,}')

    counts: Dict[Tuple[Tuple[Tuple[str, str], ...], str], Counter] = defaultdict(Counter)
    for row_no, rec in enumerate(iter_jsonl(path), 1):
        values = enrich_cargo(rec) if dataset == 'cargo' else {field: norm(rec.get(field)) for field in condition_fields}
        for combo in condition_combos:
            cond = condition_key(values, combo)
            if not cond or cond not in eligible_conditions:
                continue
            cond_fields = {field for field, _ in cond}
            for target in targets:
                if target in cond_fields:
                    continue
                target_value = values.get(target) if dataset == 'cargo' else norm(rec.get(target))
                if target_value:
                    counts[(cond, target)][target_value] += 1
        if row_no % 250_000 == 0:
            print(f'{dataset}: pass2 rows={row_no:,} counters={len(counts):,} elapsed={time.time()-started:.1f}s')

    rules = []
    for (cond, target), counter in counts.items():
        total = sum(counter.values())
        if total < min_support:
            continue
        value, support = counter.most_common(1)[0]
        confidence = support / total
        if support >= min_support and confidence >= threshold:
            rules.append({
                'dataset': dataset,
                'conditions': [{'field': field, 'value': value} for field, value in cond],
                'then_field': target,
                'then_value': value,
                'support': int(support),
                'total': int(total),
                'confidence': round(confidence, 4),
                'alternatives': [[k, int(v)] for k, v in counter.most_common(5)],
            })
    rules.sort(key=specificity_score, reverse=True)
    print(f'{dataset}: rules={len(rules):,} elapsed={time.time()-started:.1f}s')
    return rules


def keep_rules_with_lift(multirules: List[Dict[str, Any]], min_lift: float) -> List[Dict[str, Any]]:
    pair_rules = [rule for rule in multirules if len(rule['conditions']) == 2]
    best_pair = {}
    for rule in pair_rules:
        for cond in rule['conditions']:
            key = (cond['field'], cond['value'], rule['then_field'], rule['then_value'])
            best_pair[key] = max(best_pair.get(key, 0.0), rule['confidence'])

    kept = []
    for rule in multirules:
        base = 0.0
        for cond in rule['conditions']:
            key = (cond['field'], cond['value'], rule['then_field'], rule['then_value'])
            base = max(base, best_pair.get(key, 0.0))
        rule['best_single_condition_confidence'] = round(base, 4)
        rule['confidence_lift'] = round(rule['confidence'] - base, 4)
        if len(rule['conditions']) == 2 or rule['confidence_lift'] >= min_lift:
            kept.append(rule)
    return kept


def build_markdown(result: Dict[str, Any]) -> str:
    lines = ['# Colab Full Multi-field Logic Rules', '', f"- Generated: `{result['generated_at']}`", '']
    for key, title in [('declaration_rules', 'Declaration'), ('cargo_rules', 'Cargo')]:
        lines.extend([f'## {title}', '', '| Conditions | Then | Support | Confidence | Lift |', '|---|---|---:|---:|---:|'])
        for rule in result[key][:200]:
            cond = ' + '.join(f"`{c['field']}={c['value']}`" for c in rule['conditions'])
            then = f"`{rule['then_field']}={rule['then_value']}`"
            lines.append(f"| {cond} | {then} | {rule['support']:,}/{rule['total']:,} | {rule['confidence']:.2%} | {rule.get('confidence_lift', 0):.2%} |")
        lines.append('')
    return '\n'.join(lines)

print('Configured paths:')
print('DECL_PATH=', DECL_PATH)
print('CARGO_PATH=', CARGO_PATH)
print('OUT_DIR=', OUT_DIR)


## 3. Chạy full mining
Cell này có thể chạy lâu. Colab Pro/High-RAM sẽ phù hợp hơn cho `MAX_ARITY=3`.


In [ ]:

assert DECL_PATH.exists(), f'Missing declarations file: {DECL_PATH}'
assert CARGO_PATH.exists(), f'Missing cargo file: {CARGO_PATH}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

started = time.time()
decl_rules = mine_rules(DECL_PATH, DECL_RULE_FIELDS, DECL_TARGETS, 'declaration', MIN_SUPPORT, CONFIDENCE, MAX_ARITY)
cargo_rules = mine_rules(CARGO_PATH, CARGO_RULE_FIELDS, CARGO_TARGETS, 'cargo', MIN_SUPPORT, CONFIDENCE, MAX_ARITY)

result = {
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'elapsed_seconds': round(time.time() - started, 2),
    'params': {
        'min_support': MIN_SUPPORT,
        'confidence': CONFIDENCE,
        'max_arity': MAX_ARITY,
        'min_lift': MIN_LIFT,
    },
    'inputs': {'declarations': str(DECL_PATH), 'cargo': str(CARGO_PATH)},
    'declaration_rules': keep_rules_with_lift(decl_rules, MIN_LIFT),
    'cargo_rules': keep_rules_with_lift(cargo_rules, MIN_LIFT),
}

json_path = OUT_DIR / 'colab_multifield_rules_full.json'
md_path = OUT_DIR / 'colab_multifield_rules_full.md'
json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
md_path.write_text(build_markdown(result), encoding='utf-8')

print(json.dumps({
    'json': str(json_path),
    'markdown': str(md_path),
    'elapsed_seconds': result['elapsed_seconds'],
    'declaration_rules': len(result['declaration_rules']),
    'cargo_rules': len(result['cargo_rules']),
}, ensure_ascii=False, indent=2))


## 4. Xem nhanh rule mạnh


In [ ]:

# Show top useful non-direction rules after the run.
for key in ['declaration_rules', 'cargo_rules']:
    print('\n', key)
    shown = 0
    for rule in result[key]:
        if rule['then_field'] == 'direction':
            continue
        cond = ' + '.join(f"{c['field']}={c['value']}" for c in rule['conditions'])
        print(f"{cond} => {rule['then_field']}={rule['then_value']} | {rule['support']}/{rule['total']} | conf={rule['confidence']} | lift={rule.get('confidence_lift')}")
        shown += 1
        if shown >= 30:
            break
